# Spectral graph theory — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(x): return 1/(1+np.exp(-x))
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def draw_graph(A, pos=None, values=None, title=''):
    A=np.asarray(A); n=A.shape[0]
    if pos is None:
        ang=np.linspace(0,2*np.pi,n,endpoint=False); pos=np.c_[np.cos(ang),np.sin(ang)]
    plt.figure(figsize=(4,4))
    for i in range(n):
        for j in range(i+1,n):
            if A[i,j]!=0: plt.plot([pos[i,0],pos[j,0]],[pos[i,1],pos[j,1]],c='0.75',lw=1+2*abs(A[i,j]))
    c=values if values is not None else np.arange(n)
    plt.scatter(pos[:,0],pos[:,1],c=c,s=320,cmap='viridis',edgecolor='k',zorder=3)
    for i,(x,y) in enumerate(pos): plt.text(x,y,str(i),ha='center',va='center',color='white',weight='bold')
    plt.axis('off'); plt.title(title)
print('setup ok')

## The Laplacian turns connectivity into eigenvalues and eigenvectors
Spectral graph theory studies a graph through L=D-A. The examples compute its eigenvalues, use the Fiedler vector for a cut, compare normalized spectra, and show why smooth signals have small graph energy.

In [ ]:
# 1) combinatorial Laplacian spectrum of a 4-node path
A=np.array([[0,1,0,0],[1,0,1,0],[0,1,0,1],[0,0,1,0]],float); D=np.diag(A.sum(1)); L=D-A
w,V=np.linalg.eigh(L)
plt.figure(figsize=(5,3)); plt.bar(range(4),w); plt.title('path Laplacian eigenvalues')
assert np.allclose(w,[0,0.58578644,2,3.41421356])

In [ ]:
# 2) the Fiedler vector gives a two-way cut
A=np.array([[0,1,0,0],[1,0,1,0],[0,1,0,1],[0,0,1,0]],float); L=np.diag(A.sum(1))-A
w,V=np.linalg.eigh(L); f=V[:,1]; cut=(f>0).astype(int)
draw_graph(A,values=cut,title='Fiedler signs split the path')
assert cut.tolist() in ([0,0,1,1],[1,1,0,0])

In [ ]:
# 3) normalized Laplacian keeps the spectrum in [0,2]
A=np.array([[0,1,0,0],[1,0,1,0],[0,1,0,1],[0,0,1,0]],float); d=A.sum(1)
Ln=np.eye(4)-np.diag(1/np.sqrt(d))@A@np.diag(1/np.sqrt(d)); wn=np.linalg.eigvalsh(Ln)
plt.figure(figsize=(5,3)); plt.bar(range(4),wn); plt.ylim(0,2.2); plt.title('normalized spectrum')
assert np.allclose(wn,[0,0.5,1.5,2])

In [ ]:
# 4) graph energy x^T L x equals squared edge differences
A=np.array([[0,1,0,0],[1,0,1,0],[0,1,0,1],[0,0,1,0]],float); L=np.diag(A.sum(1))-A
x=np.array([1.,1.,3.,3.]); energy=x@L@x
plt.figure(figsize=(5,3)); plt.plot(x,'-o'); plt.title(f'signal energy = {energy:.1f}')
assert abs(energy-4.0)<1e-9

In [ ]:
# 5) low-pass spectral filtering damps high-frequency coefficients
A=np.array([[0,1,0,0],[1,0,1,0],[0,1,0,1],[0,0,1,0]],float); L=np.diag(A.sum(1))-A; w,V=np.linalg.eigh(L)
x=np.array([1.,-1.,1.,-1.]); coeff=V.T@x; filt=V@(coeff/(1+w))
plt.figure(figsize=(5,3)); plt.plot(x,'--o',label='raw'); plt.plot(filt,'-o',label='filtered'); plt.legend(); plt.title('low-pass smoothing')
assert np.var(filt)<np.var(x)